# insurance-demand

**Conversion, retention, and causal price elasticity modelling for UK personal lines.**

You can have a perfectly calibrated GLM and still be pricing wrong commercially. Technical price tells you what the risk is worth; the demand model tells you whether the customer will pay it. This notebook shows how naive conversion models produce badly wrong elasticities, and how the correct price/technical-premium normalisation fixes it.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burning-cost/insurance-demand/blob/main/notebooks/quickstart.ipynb)

In [ ]:
!pip install -q insurance-demand polars numpy

## Synthetic PCW quote data

We use the built-in `generate_conversion_data()` to produce a realistic PCW quote panel.
The key design: high-risk vehicles (vehicle_group 4) get higher technical premiums AND higher commercial loadings, creating confounding. A naive model conflates risk-driven price differences with commercial elasticity.

In [ ]:
from insurance_demand.datasets import generate_conversion_data, generate_retention_data

df_quotes   = generate_conversion_data(n_quotes=10_000)
df_renewals = generate_retention_data(n_policies=5_000)

print("Quotes dataset:")
print(df_quotes.head(3))
print(f"Conversion rate: {df_quotes['converted'].mean():.2%}")
print()
print("Renewals dataset:")
print(df_renewals.head(3))
print(f"Lapse rate: {df_renewals['lapsed'].mean():.2%}")

## Conversion model

`ConversionModel` uses `log(quoted_price / technical_premium)` as the price treatment — the commercial loading, not the absolute price. This is the key correction: variation in this ratio (driven by quarterly rate reviews) is approximately exogenous, unlike variation in the absolute price.

In [ ]:
from insurance_demand import ConversionModel

conv_model = ConversionModel(
    base_estimator="logistic",
    feature_cols=["age", "vehicle_group", "ncd_years", "area", "channel"],
    rank_position_col="rank_position",
)
conv_model.fit(df_quotes)
probs = conv_model.predict_proba(df_quotes)

print(f"Predicted conversion range: {probs.min():.3f} – {probs.max():.3f}")
print(f"Mean predicted: {probs.mean():.3f}, Actual: {df_quotes['converted'].mean():.3f}")

## Retention model

In [ ]:
from insurance_demand import RetentionModel

retention_model = RetentionModel(
    model_type="logistic",
    feature_cols=["tenure_years", "ncd_years", "payment_method", "claim_last_3yr"],
)
retention_model.fit(df_renewals)
lapse_probs = retention_model.predict_proba(df_renewals)

print(f"Predicted lapse range: {lapse_probs.min():.3f} – {lapse_probs.max():.3f}")
print(f"Mean predicted lapse: {lapse_probs.mean():.3f}, Actual: {df_renewals['lapsed'].mean():.3f}")

## Demand curve and optimal price

`DemandCurve` converts an elasticity estimate into a price-to-probability curve. `OptimalPrice` finds the profit-maximising price subject to the ENBP ceiling (FCA PS21/5).

In [ ]:
from insurance_demand import DemandCurve, OptimalPrice

# Build a demand curve from an elasticity estimate
# (In production this comes from ElasticityEstimator; here we use -1.5 as a placeholder)
curve = DemandCurve(
    elasticity=-1.5,
    base_price=500.0,
    base_prob=0.12,
    functional_form="semi_log",
)
prices, probs = curve.evaluate(price_range=(300, 900), n_points=60)

print("Demand curve sample:")
import polars as pl
dc_df = pl.DataFrame({"price": prices, "conv_prob": [round(p, 4) for p in probs]})
print(dc_df.filter(pl.col("price").is_in([350, 450, 500, 600, 750])).head(10))

# Find optimal price
opt = OptimalPrice(
    demand_curve=curve,
    expected_loss=380.0,
    expense_ratio=0.15,
    enbp=650.0,  # FCA PS21/5 ceiling for this renewal segment
)
result = opt.optimise()
print(f"\nOptimal price: £{result.optimal_price:.2f}")
print(f"Conversion prob: {result.conversion_prob:.4f}")
print(f"Expected profit: £{result.expected_profit:.2f}")
print(f"Constraints active: {result.constraints_active}")

## ENBP compliance audit

In [ ]:
from insurance_demand.compliance import ENBPChecker

checker = ENBPChecker()
report = checker.check(df_renewals)
print(report)

## What you should see

- `ConversionModel` with the price/tech_prem treatment recovers a price coefficient close to the true elasticity (the synthetic DGP uses −2.0). Naive log(quoted_price) would give a coefficient around −0.4 — factor-of-5 error.
- `OptimalPrice` returns £650 (the ENBP ceiling) when the unconstrained optimum lands above it — this is correct FCA-compliant behaviour.
- `ENBPChecker` on the synthetic renewal data should report zero breaches.

## Next steps

- **`ElasticityEstimator`** (requires `pip install insurance-demand[dml]`) — DML-based causal elasticity estimation that removes confounding from the technical premium correlation
- **`ConversionModel(base_estimator="catboost")`** — non-linear conversion model for large PCW panels with interaction effects
- **`RetentionModel(model_type="cox")`** — Cox proportional hazards for multi-period CLV models
- **`price_walking_report()`** — FCA tenure-based price pattern diagnostic

**GitHub:** https://github.com/burning-cost/insurance-demand  
**PyPI:** https://pypi.org/project/insurance-demand/